# 06. CAL01 정비 시각·REMARK 상세 대조

**담당:** 정수진 · **작성자 표기:** jinny · **Issue:** #75 (상위 #65 → #38)  
**대상 설비:** P1-CR1-CAL01 · **작성일:** 2026-10-15  
**브랜치:** feat/75-maintenance-timeline

감지(DETC_DT) → 착수(STRT_DT) → 완료(END_DT)를 작업지시(WO_NO)별로 확인한다.
대기시간·작업시간·정지시간을 구분하고, 정비유형·FAIL_MODE·REMARK 원문·부품교체 기록을 연결한다.
현재 제공본의 CAL01 정비는 **44건**이다. 다른 노트북의 메모리나 결과에 의존하지 않고 독립 실행한다.

| 순서 | 확인할 것 | 남기는 근거 |
|---:|---|---|
| 1 | 원본 범위·키·시각·결측 | 품질 점검표, 원문 보존 확인 |
| 2 | 44건 작업별 시각·내용·교체 부품 | 전체 작업표 4쪽, 부품교체 상세 |
| 3 | 연간 감지·착수·완료와 세 시간 길이 | 전체 타임라인, 시간 길이 비교 |
| 4 | 장기 대기·긴 정지·정지 0·동시 작업·교체 사례 | 사례 선정표, 실제 날짜·시각 확대 |
| 5 | 무엇을 확인했고 무엇이 남는가 | 검증 결과, 후속 분석 연결 |

**시간 해석:** 대기시간 = STRT_DT − DETC_DT, 작업시간 = END_DT − STRT_DT.
DOWN_MIN은 원본에 기록된 정지 **길이**이며 정확한 정지 시작·종료 시각은 없다.
작업 구간을 정지 구간으로 바꾸거나 STRT_DT + DOWN_MIN을 정지 종료 시각으로 만들지 않는다.
TBM의 DETC_DT도 값이 있으면 원본 시각으로 남기며 실제 고장 감지라고 단정하지 않는다.


## 1. 독립 실행 설정과 원본

사용 파일은 정비이력·설비마스터·부품교체이력·부품마스터 네 개다.
CSV는 문자열로 먼저 읽어 REMARK의 공백·오타·빈 문자열과 FAIL_MODE 결측을 보존한다.
시각과 수치는 별도 분석 컬럼으로 변환하고 변환 실패를 숨기지 않는다.
원본 CSV와 파생 CSV는 저장하거나 커밋하지 않는다. 그림은 figures/06_maintenance_timeline_jinny/에 저장한다.


In [ ]:
from pathlib import Path
import json
import math
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "data").is_dir() else HERE.parent
if not (ROOT / "data").is_dir() or not (ROOT / "notebooks").is_dir():
    raise FileNotFoundError("저장소 루트 또는 notebooks 폴더에서 실행하세요.")
DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "figures" / "06_maintenance_timeline_jinny"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ENC = "utf-8-sig"
EQP_TAG = "P1-CR1-CAL01"

installed = {font.name for font in font_manager.fontManager.ttflist}
for candidate in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if candidate in installed:
        rcParams["font.family"] = candidate
        break
else:
    print("한글 폰트가 없습니다. 한글 폰트 설치 후 그림을 다시 생성하세요.")
rcParams.update({
    "axes.unicode_minus": False, "font.size": 10.5,
    "axes.titlesize": 12, "axes.labelsize": 10.5,
    "figure.dpi": 105, "savefig.dpi": 155,
})
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 35)
pd.set_option("display.max_colwidth", None)
COLORS = {"BM": "#c34b47", "CBM": "#007f8b", "TBM": "#4e6ba6"}
WAIT_COLOR, DOWN_COLOR = "#7b8490", "#84559c"
saved_figures = []

def read_raw(filename, required):
    path = DATA_DIR / filename
    frame = pd.read_csv(path, encoding=ENC, dtype=str, keep_default_na=False)
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"{filename}: 필요한 컬럼 누락 {sorted(missing)}")
    return frame

def save_figure(fig, filename):
    path = FIG_DIR / filename
    if path.exists() and path.suffix != ".png":
        raise ValueError("그림 저장 경로를 확인하세요.")
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    saved_figures.append(filename)
    plt.show()
    plt.close(fig)

def actual_clock(ax, minticks=4, maxticks=8):
    locator = mdates.AutoDateLocator(minticks=minticks, maxticks=maxticks)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
    ax.set_xlabel("실제 기록 날짜·시각 (2024년, 원본 표기)")
    ax.grid(axis="x", alpha=.2)
    ax.spines[["top", "right"]].set_visible(False)

def number(value):
    return "미확인" if pd.isna(value) else f"{value:,.0f}"

def timestamp(value):
    return "미기재/변환실패" if pd.isna(value) else value.strftime("%Y-%m-%d %H:%M")

def literal(value):
    # JSON 따옴표로 원문 공백과 빈 문자열도 눈에 보이게 한다. 정제값으로 덮어쓰지 않는다.
    return json.dumps(str(value), ensure_ascii=False)


In [ ]:
maintenance_raw = read_raw("G-02_정비이력.csv", [
    "WO_NO", "EQP_TAG", "DETC_DT", "STRT_DT", "END_DT", "MNT_TYPE",
    "FAIL_MODE", "DOWN_MIN", "PART_CHG", "REMARK",
])
equipment_raw = read_raw("G-01_설비마스터.csv", ["EQP_TAG", "EQP_NM", "EQP_TYPE"])
changes_raw = read_raw("G-02_부품교체이력.csv", ["WO_NO", "PART_CD", "QTY", "CHG_RSN"])
part_master_raw = read_raw("G-01_부품마스터.csv", ["PART_CD", "PART_NM", "PART_TYPE"])

assert maintenance_raw.WO_NO.ne("").all() and maintenance_raw.WO_NO.is_unique, "정비 WO 키 확인"
assert equipment_raw.EQP_TAG.ne("").all() and equipment_raw.EQP_TAG.is_unique, "설비 키 확인"
assert part_master_raw.PART_CD.ne("").all() and part_master_raw.PART_CD.is_unique, "부품 키 확인"
equipment = equipment_raw.loc[equipment_raw.EQP_TAG.eq(EQP_TAG)]
assert len(equipment) == 1, "CAL01 설비마스터가 정확히 한 행이어야 합니다."
cal_raw = maintenance_raw.loc[maintenance_raw.EQP_TAG.eq(EQP_TAG)].copy()
assert len(cal_raw) == 44, "현재 제공본의 CAL01 44건과 다릅니다. 원본 범위를 확인하세요."

cal = cal_raw.copy()
for column in ["DETC_DT", "STRT_DT", "END_DT"]:
    cal[column + "_RAW"] = cal[column]
    cal[column] = pd.to_datetime(cal[column], format="%Y-%m-%d %H:%M", errors="coerce")
cal["DOWN_MIN_RAW"] = cal.DOWN_MIN
cal["DOWN_MIN"] = pd.to_numeric(cal.DOWN_MIN, errors="coerce")
cal["WAIT_MIN"] = (cal.STRT_DT - cal.DETC_DT).dt.total_seconds() / 60
cal["WORK_MIN"] = (cal.END_DT - cal.STRT_DT).dt.total_seconds() / 60
cal["DETECT_TO_END_MIN"] = (cal.END_DT - cal.DETC_DT).dt.total_seconds() / 60
cal["REMARK_EMPTY"] = cal.REMARK.eq("")
cal["REMARK_WHITESPACE"] = cal.REMARK.ne(cal.REMARK.str.strip())
cal = cal.sort_values(["STRT_DT", "WO_NO"], na_position="last").reset_index(drop=True)
cal["ORDER"] = np.arange(1, len(cal) + 1)

# 원행을 남긴다. 중복을 임의 삭제하거나 수량을 합산하지 않는다.
changes = changes_raw.copy()
changes["SOURCE_CSV_ROW"] = changes.index + 2
changes["EXACT_DUPLICATE"] = changes_raw.duplicated(keep=False)
cal_parts = changes.loc[changes.WO_NO.isin(cal.WO_NO)].merge(
    part_master_raw[["PART_CD", "PART_NM", "PART_TYPE"]],
    on="PART_CD", how="left", validate="many_to_one", indicator="MASTER_LINK",
)
cal_parts["PART_NM"] = cal_parts.PART_NM.fillna("부품마스터 미등록")
cal_parts["PART_TYPE"] = cal_parts.PART_TYPE.fillna("미등록")
cal_parts = cal_parts.sort_values(["WO_NO", "SOURCE_CSV_ROW"]).reset_index(drop=True)
part_counts = cal_parts.groupby("WO_NO").size()
cal["PART_ROWS"] = cal.WO_NO.map(part_counts).fillna(0).astype(int)
cal["PART_FLAG_MISMATCH"] = cal.PART_CHG.eq("Y").ne(cal.PART_ROWS.gt(0))

def parts_text(wo):
    rows = cal_parts.loc[cal_parts.WO_NO.eq(wo)]
    if rows.empty:
        return "교체 기록 없음"
    return "\n".join(
        f"{r.PART_CD} {r.PART_NM} ×{r.QTY} / {r.CHG_RSN}"
        + (" [원행 중복]" if r.EXACT_DUPLICATE else "")
        for r in rows.itertuples()
    )

cal["PARTS_DETAIL"] = cal.WO_NO.map(parts_text)
display(equipment[["EQP_TAG", "EQP_NM", "EQP_TYPE"]])
display(pd.DataFrame({
    "원본": ["정비이력 전체", "CAL01 정비", "CAL01 부품교체 원행", "CAL01 교체 WO"],
    "건수": [len(maintenance_raw), len(cal), len(cal_parts), cal_parts.WO_NO.nunique()],
}))


## 2. 시각·결측·연결 점검

시각 파싱 실패·음수 대기/작업·DOWN_MIN 결측·교체 여부와 실제 연결의 불일치를 별도로 확인한다.
FAIL_MODE와 REMARK의 빈 값은 채우지 않는다. TBM에도 감지 시각 또는 고장모드가 적힐 수 있으므로,
데이터사전의 일반 설명보다 실제 원행을 먼저 확인한다.
정비지시와 부품교체는 일대다 관계다. 작업표는 WO당 한 행을 유지하고 교체 상세는 별도 표로 남긴다.


In [ ]:
audit = pd.DataFrame([
    ("CAL01 정비", len(cal), "현재 제공본 기준 44건"),
    ("DETC_DT 미기재", int(cal.DETC_DT_RAW.eq("").sum()), "결측은 대기시간 미확인"),
    ("DETC_DT 변환 실패", int((cal.DETC_DT_RAW.ne("") & cal.DETC_DT.isna()).sum()), "원문 확인 필요"),
    ("STRT_DT 미확인", int(cal.STRT_DT.isna().sum()), "작업 시작 추정 금지"),
    ("END_DT 미확인", int(cal.END_DT.isna().sum()), "작업 종료 추정 금지"),
    ("음수 대기시간", int(cal.WAIT_MIN.lt(0).sum()), "원시각 순서 확인"),
    ("음수 작업시간", int(cal.WORK_MIN.lt(0).sum()), "원시각 순서 확인"),
    ("DOWN_MIN 미확인", int(cal.DOWN_MIN.isna().sum()), "0으로 채우지 않음"),
    ("DOWN_MIN 음수", int(cal.DOWN_MIN.lt(0).sum()), "원본 확인 필요"),
    ("DOWN_MIN = 0", int(cal.DOWN_MIN.eq(0).sum()), "기록값 0; 구체적 운전상태는 미확정"),
    ("DOWN_MIN > 작업시간", int(cal.DOWN_MIN.gt(cal.WORK_MIN).sum()), "서로 다른 시간 정의; 자동 보정 금지"),
    ("FAIL_MODE 빈 문자열", int(cal.FAIL_MODE.eq("").sum()), "고장모드로 채우지 않음"),
    ("REMARK 빈 문자열", int(cal.REMARK_EMPTY.sum()), "원문 빈 값 보존"),
    ("REMARK 앞뒤 공백", int(cal.REMARK_WHITESPACE.sum()), "원문 공백 보존"),
    ("교체 표기·연결 불일치", int(cal.PART_FLAG_MISMATCH.sum()), "PART_CHG와 WO 연결 비교"),
    ("부품마스터 미연결", int(cal_parts.MASTER_LINK.ne("both").sum()), "부품코드 그대로 표시"),
    ("CAL01 교체 원행 중복", int(cal_parts.EXACT_DUPLICATE.sum()), "원행 유지; 수량 합산하지 않음"),
], columns=["점검", "건수", "처리·해석"])
display(audit)
type_quality = cal.groupby("MNT_TYPE", dropna=False).agg(
    정비건수=("WO_NO", "size"),
    감지시각존재=("DETC_DT", "count"),
    고장모드빈값=("FAIL_MODE", lambda s: s.eq("").sum()),
    REMARK빈값=("REMARK_EMPTY", "sum"),
    부품교체WO=("PART_ROWS", lambda s: s.gt(0).sum()),
)
display(type_quality)
issues = cal.loc[
    cal.STRT_DT.isna() | cal.END_DT.isna()
    | cal.WAIT_MIN.lt(0) | cal.WORK_MIN.lt(0)
    | cal.DOWN_MIN.isna() | cal.DOWN_MIN.lt(0) | cal.PART_FLAG_MISMATCH
]
display(issues[["WO_NO", "DETC_DT_RAW", "STRT_DT_RAW", "END_DT_RAW",
                "DOWN_MIN_RAW", "PART_CHG", "PART_ROWS"]])


## 3. 전체 작업표와 부품교체 상세

**읽는 순서:** 착수 순번 → 세 원시각 → 대기·작업·정지 길이 → REMARK → 교체 기록.
원문은 JSON 따옴표로 감싸 빈 값은 ""로, 앞뒤 공백은 따옴표 안 공백으로 표시한다.
행을 생략하지 않고 12건씩 그림으로 저장한다. 한 WO의 여러 부품을 펼쳐서 정비 건수가 늘어나지 않게 한다.
정비유형 BM/CBM/TBM의 색은 종류만 구분하며 고장 심각도나 정상 판정이 아니다.


In [ ]:
work_table = cal[[
    "ORDER", "WO_NO", "MNT_TYPE", "FAIL_MODE",
    "DETC_DT", "STRT_DT", "END_DT", "WAIT_MIN", "WORK_MIN", "DOWN_MIN",
    "PART_CHG", "REMARK", "PARTS_DETAIL",
]].copy()
display(work_table)

def table_page(page, page_no, page_count):
    labels = [
        "순번 / WO_NO\n정비 / FAIL_MODE", "감지 → 착수 → 완료\n실제 날짜·시각",
        "대기 / 작업 / 정지\n각각 분", "REMARK 원문\n따옴표 안 공백도 원문",
        "교체 표기 / 부품·수량·사유\n실제 교체 시각은 제공되지 않음",
    ]
    values = []
    for row in page.itertuples():
        mode = row.FAIL_MODE if row.FAIL_MODE else "(빈 값)"
        remark = "\n".join(textwrap.wrap(literal(row.REMARK), width=26,
                    replace_whitespace=False, drop_whitespace=False)) or '""'
        parts = "\n".join(
            "\n".join(textwrap.wrap(line, width=33, replace_whitespace=False,
                                    drop_whitespace=False))
            for line in row.PARTS_DETAIL.splitlines()
        )
        values.append([
            f"{row.ORDER:02d} · {row.WO_NO}\n{row.MNT_TYPE} / {mode}",
            f"감지 {timestamp(row.DETC_DT)}\n착수 {timestamp(row.STRT_DT)}\n완료 {timestamp(row.END_DT)}",
            f"대기 {number(row.WAIT_MIN)}\n작업 {number(row.WORK_MIN)}\n정지 {number(row.DOWN_MIN)}",
            remark, f"PART_CHG={row.PART_CHG}\n{parts}",
        ])
    heights = [max(4, max(value.count("\n") + 1 for value in row)) for row in values]
    fig, ax = plt.subplots(figsize=(22, 1.7 + .22 * sum(heights)))
    ax.axis("off")
    fig.suptitle(f"CAL01 전체 작업표 · {page_no}/{page_count} · 착수 순서", x=.02, ha="left",
                 fontsize=19, fontweight="bold", y=.985)
    fig.text(.02, .947, "감지·착수·완료는 원시각 | 정지는 DOWN_MIN 길이만 확인 | REMARK·교체 원행 보존",
             fontsize=11, color="#485563")
    table = ax.table(cellText=values, colLabels=labels, cellLoc="left", colLoc="left",
                     colWidths=[.17, .24, .12, .20, .27], bbox=[0, 0, 1, .96])
    table.auto_set_font_size(False)
    table.set_fontsize(9.5)
    total_height = sum(heights) + 4
    for (ri, ci), cell in table.get_celld().items():
        cell.set_edgecolor("#d6dde3")
        cell.PAD = .045
        if ri == 0:
            cell.set_facecolor("#e6edf3")
            cell.set_text_props(weight="bold")
            cell.set_height(4 / total_height)
        else:
            cell.set_height(heights[ri - 1] / total_height)
            cell.set_facecolor("#f5f7fa" if ri % 2 == 0 else "white")
            if ci == 0:
                cell.get_text().set_color(COLORS.get(page.iloc[ri - 1].MNT_TYPE, "#333333"))
    fig.subplots_adjust(left=.02, right=.98, top=.91, bottom=.02)
    save_figure(fig, f"01_work_table_{page_no:02d}.png")

page_size = 12
page_count = math.ceil(len(cal) / page_size)
table_covered = []
for page_no, start in enumerate(range(0, len(cal), page_size), 1):
    page = cal.iloc[start:start + page_size]
    table_covered.extend(page.WO_NO.tolist())
    table_page(page, page_no, page_count)
assert table_covered == cal.WO_NO.tolist(), "44건 작업표의 중복·누락 확인"
display(cal_parts[["SOURCE_CSV_ROW", "WO_NO", "PART_CD", "PART_NM",
                   "PART_TYPE", "QTY", "CHG_RSN", "EXACT_DUPLICATE", "MASTER_LINK"]])


## 4. 연간 감지 → 착수 → 완료

왼쪽은 실제 달력 시각이다. 회색 점선은 감지~착수 대기, 정비유형 색 실선은 착수~완료 작업이다.
마름모·원·사각형으로 감지·착수·완료를 구분한다. 같은 시각처럼 보이는 짧은 구간은 사례 그림에서 확대한다.
오른쪽은 같은 WO의 대기·작업·정지 **길이**이며 달력 축과 독립적이다. 정지시간을 왼쪽 구간에 덧그리지 않는다.


In [ ]:
fig, (ax, duration_ax) = plt.subplots(
    1, 2, figsize=(23, 23), sharey=True, gridspec_kw={"width_ratios": [3.3, 1.2]})
fig.subplots_adjust(left=.14, right=.98, top=.94, bottom=.065, wspace=.10)
fig.suptitle("CAL01 · 44건의 정비 시각과 시간 길이", x=.02, ha="left",
             fontsize=22, fontweight="bold", y=.988)
fig.text(.02, .965, "왼쪽: 2024년 실제 기록 시각 | 오른쪽: 원점 없는 길이 비교 | 정지 시작·종료 시각은 미제공",
         fontsize=12, color="#485563")
y = np.arange(len(cal))
for i, row in enumerate(cal.itertuples()):
    color = COLORS.get(row.MNT_TYPE, "#555555")
    for target in (ax, duration_ax):
        if i % 2 == 0:
            target.axhspan(i - .5, i + .5, color="#f3f6f9", zorder=0)
    if pd.notna(row.DETC_DT) and pd.notna(row.STRT_DT):
        ax.plot([row.DETC_DT, row.STRT_DT], [i, i], "--", lw=1.5, color=WAIT_COLOR)
    if pd.notna(row.STRT_DT) and pd.notna(row.END_DT):
        ax.plot([row.STRT_DT, row.END_DT], [i, i], lw=4, color=color)
    for field, marker in [("DETC_DT", "D"), ("STRT_DT", "o"), ("END_DT", "s")]:
        value = getattr(row, field)
        if pd.notna(value):
            ax.scatter([value], [i], marker=marker, s=23, color=color, zorder=4)
    for offset, field, marker, color in [
        (-.19, "WAIT_MIN", "D", WAIT_COLOR), (0, "WORK_MIN", "s", COLORS[row.MNT_TYPE]),
        (.19, "DOWN_MIN", "o", DOWN_COLOR),
    ]:
        value = getattr(row, field)
        if pd.notna(value):
            duration_ax.scatter(value, i + offset, marker=marker, color=color, s=27, zorder=4)
ax.set_yticks(y, [f"{r.ORDER:02d} {r.WO_NO}\n{r.MNT_TYPE} · {r.FAIL_MODE or '빈 값'}"
                 for r in cal.itertuples()], fontsize=9)
ax.set_ylim(len(cal) - .4, -.6)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
ax.set_xlim(pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01"))
ax.set_xlabel("실제 기록 날짜·시각 · 2024년")
ax.set_ylabel("착수 순번 · 작업지시번호 · 정비유형 · 고장모드")
ax.grid(axis="x", alpha=.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(handles=[
    Line2D([], [], color=WAIT_COLOR, linestyle="--", label="대기: 감지~착수"),
    *[Line2D([], [], color=color, lw=4, label=f"{kind} 작업") for kind, color in COLORS.items()],
    *[Line2D([], [], color="#444444", marker=marker, linestyle="", label=label)
      for marker, label in [("D", "감지"), ("o", "착수"), ("s", "완료")]]],
    loc="lower left", bbox_to_anchor=(0, 1.005), ncol=4, fontsize=9, frameon=False)
duration_ax.set_xscale("symlog", linthresh=60)
duration_ax.set_xticks([0, 60, 300, 1000, 5000, 15000], ["0", "60", "300", "1,000", "5,000", "15,000"])
duration_ax.set_xlim(-8, max(cal.WAIT_MIN.max(), cal.WORK_MIN.max(), cal.DOWN_MIN.max()) * 1.25)
duration_ax.set_xlabel("길이 (분)\n0~60분 선형, 그 이상 로그 축")
duration_ax.tick_params(axis="y", left=False, labelleft=False)
duration_ax.grid(axis="x", alpha=.25)
duration_ax.spines[["top", "right", "left"]].set_visible(False)
duration_ax.legend(handles=[
    Line2D([], [], color=color, marker=marker, linestyle="", label=label)
    for color, marker, label in [(WAIT_COLOR, "D", "대기"), ("#4b5968", "s", "작업 (정비유형 색)"),
                                 (DOWN_COLOR, "o", "정지 DOWN_MIN")]],
    loc="lower left", bbox_to_anchor=(0, 1.005), fontsize=9, frameon=False)
save_figure(fig, "02_all_maintenance_timeline.png")


## 5. 대기·작업·정지 길이 비교

대기시간은 접수부터 착수까지의 달력 차이이며 설비가 그 시간 내내 정지했다는 뜻이 아니다.
작업시간과 DOWN_MIN의 일치·차이를 점선(같은 길이)과 대조한다.
정지시간을 모두 더해 설비의 연간 실제 정지 총량이라고 해석하지 않는다.
작업끼리 겹치는 경우도 별도 확인하므로 단순 작업시간 합계는 실제 점유 시간과 다르다.


In [ ]:
duration_summary = cal.groupby("MNT_TYPE").agg(
    건수=("WO_NO", "size"), 대기중앙값분=("WAIT_MIN", "median"),
    대기최대분=("WAIT_MIN", "max"), 작업중앙값분=("WORK_MIN", "median"),
    작업최대분=("WORK_MIN", "max"), 정지중앙값분=("DOWN_MIN", "median"),
    정지최대분=("DOWN_MIN", "max"), 정지0건수=("DOWN_MIN", lambda s: s.eq(0).sum()),
)
display(duration_summary)
fig, axs = plt.subplots(1, 2, figsize=(17, 7))
fig.subplots_adjust(left=.07, right=.97, top=.80, bottom=.14, wspace=.26)
fig.suptitle("CAL01 · 대기시간, 작업시간, 정지시간은 서로 다른 값", x=.03, ha="left",
             fontsize=19, fontweight="bold", y=.97)
fig.text(.03, .90, "각 점은 WO 1건 | 유형 색은 동일 | 정지 0도 원본값으로 유지", fontsize=11)
for kind, group in cal.groupby("MNT_TYPE"):
    axs[0].scatter(group.WAIT_MIN, group.WORK_MIN, label=f"{kind} ({len(group)}건)",
                   s=45, color=COLORS.get(kind, "#555555"), alpha=.8, edgecolor="white", linewidth=.4)
    axs[1].scatter(group.WORK_MIN, group.DOWN_MIN, s=45,
                   color=COLORS.get(kind, "#555555"), alpha=.8, edgecolor="white", linewidth=.4)
axs[0].set_xscale("symlog", linthresh=60)
axs[0].set_xlabel("감지~착수 대기시간 (분; 60분 이후 로그 축)")
axs[0].set_ylabel("착수~완료 작업시간 (분)")
axs[0].set_title("대기가 길어도 정지시간의 위치·길이는 별개", loc="left")
limit = max(cal.WORK_MIN.max(), cal.DOWN_MIN.max()) * 1.08
axs[1].plot([0, limit], [0, limit], "--", color=WAIT_COLOR, lw=1, label="작업시간 = DOWN_MIN")
axs[1].set(xlim=(-25, limit), ylim=(-25, limit),
           xlabel="착수~완료 작업시간 (분)", ylabel="정지시간 DOWN_MIN (분)")
axs[1].set_title("작업시간과 정지 길이가 같은지 대조", loc="left")
for ax in axs:
    ax.grid(alpha=.2)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, loc="upper left")
for wo, reason in [(cal.loc[cal.WAIT_MIN.idxmax(), "WO_NO"], "최장 대기"),
                   (cal.loc[cal.DOWN_MIN.idxmax(), "WO_NO"], "최대 정지")]:
    row = cal.loc[cal.WO_NO.eq(wo)].iloc[0]
    target, x, yval = (axs[0], row.WAIT_MIN, row.WORK_MIN) if reason == "최장 대기" else (axs[1], row.WORK_MIN, row.DOWN_MIN)
    target.annotate(f"{reason}\n{wo}", (x, yval), xytext=(-8, 18), textcoords="offset points",
                    ha="right", fontsize=9, arrowprops={"arrowstyle":"-", "color":"#465563"})
save_figure(fig, "03_wait_work_down_comparison.png")


## 6. 동시 작업과 관심 사례 선정

같은 설비의 작업 구간을 **[착수, 완료)**로 비교한다. 한 작업 완료와 다음 착수가 같은 경우는 겹침으로 세지 않는다.
감지 시각이 작업 밖에 있다는 이유로 다른 WO와 합치지 않는다.
확대 사례는 최대 대기·최대 정지·정지 0 중 최장 작업·복수 부품교체·모든 동시 작업 쌍·
7월 절연 저하 기록·12월 교체 기록을 명시적인 규칙으로 선택한다.


In [ ]:
overlap_rows = []
for i, left in cal.iterrows():
    for _, right in cal.iloc[i + 1:].iterrows():
        if pd.isna(left.STRT_DT) or pd.isna(left.END_DT) or pd.isna(right.STRT_DT) or pd.isna(right.END_DT):
            continue
        start, end = max(left.STRT_DT, right.STRT_DT), min(left.END_DT, right.END_DT)
        if end > start:
            overlap_rows.append({
                "WO_A": left.WO_NO, "WO_B": right.WO_NO, "OVERLAP_START": start,
                "OVERLAP_END": end, "OVERLAP_MIN": (end - start).total_seconds() / 60,
            })
overlaps = pd.DataFrame(overlap_rows, columns=[
    "WO_A", "WO_B", "OVERLAP_START", "OVERLAP_END", "OVERLAP_MIN"])
display(overlaps)

cases = []
def add_case(key, title, wo_numbers, reason):
    selected = cal.loc[cal.WO_NO.isin(wo_numbers)]
    assert len(selected) == len(set(wo_numbers)), f"사례 WO 확인: {wo_numbers}"
    cases.append({"CASE": key, "TITLE": title, "WO_LIST": list(selected.WO_NO), "REASON": reason})

add_case("A", "최장 대기", [cal.loc[cal.WAIT_MIN.idxmax(), "WO_NO"]],
         "감지~착수 대기시간 최대; 대기와 정지 길이를 분리")
add_case("B", "최대 정지 길이", [cal.loc[cal.DOWN_MIN.idxmax(), "WO_NO"]],
         "DOWN_MIN 최대; 작업시간과 정지 길이의 차이")
zero = cal.loc[cal.DOWN_MIN.eq(0)].sort_values(["WORK_MIN", "WO_NO"], ascending=[False, True])
if not zero.empty:
    add_case("C", "정지 0 중 최장 작업", [zero.iloc[0].WO_NO],
             "DOWN_MIN=0이고 작업시간이 가장 긴 WO; 운전상태 추정 금지")
multi = cal.loc[cal.PART_ROWS.gt(1)]
if not multi.empty:
    add_case("D", "한 WO의 여러 교체 부품", [multi.iloc[0].WO_NO],
             "일대다 교체 연결을 실제 부품·수량·사유로 확인")
for i, pair in enumerate(overlaps.itertuples(), 1):
    add_case(f"E{i}", "실제 작업 구간 겹침", [pair.WO_A, pair.WO_B],
             f"작업 교집합 {number(pair.OVERLAP_MIN)}분; WO별 REMARK·정비유형 분리")
for key, wo, title in [
    ("F", "WO-2024-01094", "7월 절연 저하·교체 기록"),
    ("G", "WO-2024-02105", "12월 감지·착수·교체 기록"),
]:
    if cal.WO_NO.eq(wo).any():
        add_case(key, title, [wo], "05번에서 관심을 둔 WO의 세 시각·원문·부품을 독립 대조")
case_table = pd.DataFrame([
    {"사례": c["CASE"], "주제": c["TITLE"], "대상 WO": ", ".join(c["WO_LIST"]), "선정 근거": c["REASON"]}
    for c in cases
])
display(case_table)


## 7. 관심 사례 확대: 원시각과 원문을 함께 읽기

각 사례의 위쪽 왼쪽은 감지~완료 전체, 가운데는 착수~완료 확대,
오른쪽은 대기·작업·정지 **분 단위 길이**다.
달력 축에는 날짜와 시각을 표시하며, 작업 확대에서 감지 시각이 보이지 않으면 전체 패널에서 확인한다.
두 WO가 겹치면 실제 작업 교집합만 붉은 음영으로 표시한다.
아래 카드에는 WO·정비유형·FAIL_MODE·세 원시각·REMARK·교체 부품을 모두 남긴다.
부품교체 자료에는 실제 교체 시각이 없으므로 완료 시각에 교체 사건을 만들어 찍지 않는다.


In [ ]:
def draw_task(ax, row, yval):
    color = COLORS.get(row.MNT_TYPE, "#555555")
    if pd.notna(row.DETC_DT) and pd.notna(row.STRT_DT):
        ax.plot([row.DETC_DT, row.STRT_DT], [yval, yval], "--", color=WAIT_COLOR, lw=2)
    if pd.notna(row.STRT_DT) and pd.notna(row.END_DT):
        ax.plot([row.STRT_DT, row.END_DT], [yval, yval], color=color, lw=8, solid_capstyle="butt")
    for field, marker, label in [("DETC_DT", "D", "감지"), ("STRT_DT", "o", "착수"), ("END_DT", "s", "완료")]:
        value = getattr(row, field)
        if pd.notna(value):
            ax.scatter([value], [yval], color=color, marker=marker, s=55, edgecolor="white", linewidth=.8, zorder=4)

def case_figure(case):
    selected = cal.loc[cal.WO_NO.isin(case["WO_LIST"])].copy()
    rows = list(selected.itertuples())
    count = len(rows)
    fig = plt.figure(figsize=(22, 7.7 + 1.4 * count))
    grid = fig.add_gridspec(2, 3, height_ratios=[2.3, 1.4 * count],
                            width_ratios=[1.3, 1.3, 1], hspace=.49, wspace=.34)
    whole = fig.add_subplot(grid[0, 0])
    work = fig.add_subplot(grid[0, 1])
    lengths = fig.add_subplot(grid[0, 2])
    card = fig.add_subplot(grid[1, :])
    fig.subplots_adjust(left=.09, right=.97, top=.80, bottom=.06)
    fig.suptitle(f"{case['CASE']} · CAL01 · {case['TITLE']}", x=.025, ha="left",
                 fontsize=21, fontweight="bold", y=.975)
    fig.text(.025, .922, case["REASON"], fontsize=11, color="#485563")
    valid_times = selected[["DETC_DT", "STRT_DT", "END_DT"]].stack()
    lo, hi = valid_times.min(), valid_times.max()
    work_lo, work_hi = selected.STRT_DT.min(), selected.END_DT.max()
    whole_pad = max((hi - lo) * .07, pd.Timedelta(minutes=12))
    work_pad = max((work_hi - work_lo) * .12, pd.Timedelta(minutes=12))
    for i, row in enumerate(rows):
        draw_task(whole, row, i)
        draw_task(work, row, i)
        for offset, field, color in [(-.22, "WAIT_MIN", WAIT_COLOR),
                                     (0, "WORK_MIN", COLORS[row.MNT_TYPE]),
                                     (.22, "DOWN_MIN", DOWN_COLOR)]:
            value = getattr(row, field)
            if pd.notna(value):
                lengths.barh(i + offset, value, height=.18, color=color, alpha=.9)
                lengths.annotate(number(value), (value, i + offset), xytext=(5, 0),
                                 textcoords="offset points", va="center", fontsize=9)
    for ax in (whole, work, lengths):
        ax.set_ylim(count - .45, -.55)
        ax.set_yticks(range(count), [row.WO_NO for row in rows], fontsize=9)
    work.tick_params(axis="y", labelleft=False)
    lengths.tick_params(axis="y", labelleft=False)
    whole.set_xlim(lo - whole_pad, hi + whole_pad)
    work.set_xlim(work_lo - work_pad, work_hi + work_pad)
    for ax in (whole, work):
        actual_clock(ax, 4, 6)
    whole.set_title("감지 → 착수 → 완료 전체", loc="left", pad=15)
    work.set_title("착수 → 완료 작업 확대", loc="left", pad=15)
    for pair in overlaps.itertuples():
        if pair.WO_A in case["WO_LIST"] and pair.WO_B in case["WO_LIST"]:
            for ax in (whole, work):
                ax.axvspan(pair.OVERLAP_START, pair.OVERLAP_END, color="#c34b47", alpha=.12)
    lengths.set_title("원점 없는 시간 길이", loc="left", pad=15)
    maximum = selected[["WAIT_MIN", "WORK_MIN", "DOWN_MIN"]].max().max()
    lengths.set_xlim(0, max(maximum * 1.28, 60))
    lengths.set_xlabel("길이 (분) · 정지 구간 위치는 미상")
    lengths.grid(axis="x", alpha=.2)
    lengths.spines[["top", "right"]].set_visible(False)
    fig.legend(handles=[
        Line2D([], [], color=WAIT_COLOR, linestyle="--", label="감지~착수 대기"),
        *[Line2D([], [], color=color, lw=4, label=f"{kind} 작업") for kind, color in COLORS.items()],
        Line2D([], [], color=DOWN_COLOR, lw=4, label="정지 길이 (오른쪽만)"),
        *[Line2D([], [], color="#444444", marker=marker, linestyle="", label=label)
          for marker, label in [("D", "감지"), ("o", "착수"), ("s", "완료")]]],
        loc="upper left", bbox_to_anchor=(.025, .90), ncol=8, frameon=False, fontsize=9)
    card.axis("off")
    for i, row in enumerate(rows):
        mode = row.FAIL_MODE if row.FAIL_MODE else "(빈 값)"
        lines = [
            f"{row.WO_NO} · {row.MNT_TYPE} · FAIL_MODE={mode} · PART_CHG={row.PART_CHG}",
            f"감지 {timestamp(row.DETC_DT)} → 착수 {timestamp(row.STRT_DT)} → 완료 {timestamp(row.END_DT)}",
            f"대기 {number(row.WAIT_MIN)}분 / 작업 {number(row.WORK_MIN)}분 / DOWN_MIN {number(row.DOWN_MIN)}분",
            "REMARK 원문 = " + literal(row.REMARK),
            "교체 원행: " + row.PARTS_DETAIL.replace("\n", " ; "),
        ]
        wrapped = "\n".join("\n".join(textwrap.wrap(line, width=115, replace_whitespace=False,
                           drop_whitespace=False)) for line in lines)
        card.text(.01, 1 - i / count, wrapped, transform=card.transAxes, va="top", fontsize=11,
                  linespacing=1.7, bbox={"facecolor":"#f3f6f9", "edgecolor":"#d6dde3",
                                       "boxstyle":"round,pad=.7"})
    save_figure(fig, f"04_case_{case['CASE']}.png")

for case in cases:
    display(Markdown(f"### {case['CASE']} · {case['TITLE']}\n\n{case['REASON']}"))
    case_figure(case)


## 8. 실행 결과와 검증

아래 검사는 분석 수치를 바꾸지 않는다. 44건 유지, 원문 보존, 시간 차이 계산,
부품 일대다 연결, 동시 작업 경계, 전체 표의 누락·중복, 저장 그림을 실제 값으로 확인한다.
시간 관계와 자유서술만으로 고장 원인·교체 효과·센서 회복을 판정하지 않는다.


In [ ]:
checks = []
def checked(name, condition, evidence):
    assert bool(condition), name
    checks.append({"검증": name, "결과": "통과", "근거": evidence})

checked("CAL01 범위·건수", len(cal) == 44 and cal.EQP_TAG.eq(EQP_TAG).all(), f"{len(cal)}건")
checked("WO당 작업표 한 행", cal.WO_NO.is_unique and len(work_table) == len(cal), "일대다 교체로 작업 행 증가 없음")
original = cal_raw.set_index("WO_NO")
joined = cal.set_index("WO_NO").reindex(original.index)
checked("REMARK 원문 보존", joined.REMARK.equals(original.REMARK), "공백·오타·빈 문자열 그대로")
checked("FAIL_MODE 원문 보존", joined.FAIL_MODE.equals(original.FAIL_MODE), "결측을 고장모드로 채우지 않음")
for field in ["DETC_DT", "STRT_DT", "END_DT"]:
    checked(field + " 원문 보존", joined[field + "_RAW"].equals(original[field]), "분석용 datetime과 원문을 별도 보유")
checked("시각 파싱과 시간 순서", cal[["DETC_DT", "STRT_DT", "END_DT"]].notna().all().all()
        and cal.WAIT_MIN.ge(0).all() and cal.WORK_MIN.ge(0).all(), "현재 44건의 시각을 실제 확인")
checked("대기+작업=감지~완료", np.allclose(cal.WAIT_MIN + cal.WORK_MIN, cal.DETECT_TO_END_MIN, equal_nan=True),
        "같은 원시각에서 분 단위 계산")
expected_parts = changes_raw.loc[changes_raw.WO_NO.isin(cal.WO_NO)]
checked("교체 원행 보존", len(cal_parts) == len(expected_parts), f"{len(cal_parts)}행; 임의 중복 삭제·수량 합산 없음")
checked("부품·설비 마스터 연결", cal_parts.MASTER_LINK.eq("both").all() and len(equipment) == 1,
        "EQP_TAG·PART_CD 명시적 키")
checked("교체 표기·연결 일치", not cal.PART_FLAG_MISMATCH.any(), "PART_CHG와 부품교체 WO 연결 비교")
checked("모든 실제 작업 겹침 양수", overlaps.OVERLAP_MIN.gt(0).all(), f"{len(overlaps)}쌍; [착수,완료) 경계")
checked("전체 작업표 순서·누락·중복", table_covered == cal.WO_NO.tolist(), f"{page_count}쪽, {len(table_covered)}건")
expected_figures = page_count + 2 + len(cases)
checked("그림 생성", len(saved_figures) == expected_figures
        and all((FIG_DIR / name).stat().st_size > 0 for name in saved_figures), f"{len(saved_figures)}개")
checked("정지 길이 원문과 일치", joined.DOWN_MIN.equals(pd.to_numeric(original.DOWN_MIN, errors="coerce")),
        "정지 시각을 생성하지 않고 길이만 사용")
display(pd.DataFrame(checks))

long_wait = cal.loc[cal.WAIT_MIN.idxmax()]
long_down = cal.loc[cal.DOWN_MIN.idxmax()]
run_summary = {
    "maintenance_count": len(cal), "type_counts": cal.MNT_TYPE.value_counts().to_dict(),
    "part_rows": len(cal_parts), "part_work_orders": int(cal_parts.WO_NO.nunique()),
    "zero_down_count": int(cal.DOWN_MIN.eq(0).sum()),
    "remark_empty_count": int(cal.REMARK_EMPTY.sum()),
    "remark_whitespace_count": int(cal.REMARK_WHITESPACE.sum()),
    "longest_wait_wo": long_wait.WO_NO, "longest_wait_min": float(long_wait.WAIT_MIN),
    "max_down_wo": long_down.WO_NO, "max_down_min": float(long_down.DOWN_MIN),
    "max_down_work_min": float(long_down.WORK_MIN),
    "overlap_pairs": len(overlaps),
    "overlap_details": [
        {"wo_a": row.WO_A, "wo_b": row.WO_B, "minutes": float(row.OVERLAP_MIN)}
        for row in overlaps.itertuples()
    ],
    "case_count": len(cases), "figure_count": len(saved_figures), "checks": len(checks),
    "figures": saved_figures,
}
print("06_SUMMARY " + json.dumps(run_summary, ensure_ascii=False))


## 9. 결과를 읽을 때와 다음 분석 연결

- 대기시간은 감지~착수의 기록 차이다. CBM의 긴 대기는 점검 일정과 관련될 수 있으나 사유는 원문 이상으로 확정하지 않는다.
- DOWN_MIN=0은 유효한 원본값이다. 작업시간이 길어도 설비 정지가 있었다고 단정하지 않는다.
- TBM의 빈 FAIL_MODE는 그대로 두고, TBM에 값이 적힌 사례도 정비유형만으로 무효화하지 않는다.
- REMARK의 “재발”, “확인 중”, “원인 미상”은 자유서술이다. 반복 고장의 동일성·원인을 확정하는 라벨로 사용하지 않는다.
- 교체 부품·수량·사유는 WO로 연결한 근거다. 실제 교체 시각이나 교체 효과는 이 기록만으로 알 수 없다.
- 동시에 진행된 WO의 작업시간과 DOWN_MIN을 단순 합산해 설비 실제 정지시간으로 해석하지 않는다.
- #76은 LOT 내부 인접 측정시각 차이를 따로 분석한다. #77은 이 노트북의 원시각·WO·교체 내용을 센서·조업·계획수리와 대조하되 센서 관측이 없는 작업도 유지한다.

**재현:** 저장소 Python 환경에서 이 파일을 처음부터 끝까지 실행한다. 노트북 출력은 nbstripout으로 제거하고,
검토용 결과는 아래 그림으로 공유한다. 실제 실행 결과는 검증 완료 후 이 절 아래에 남긴다.


### 실제 실행 확인 · 2026-10-15

- CAL01 정비 **44건**: BM 12건, CBM 10건, TBM 22건.
- 부품교체 **8 WO · 원행 9행**. 한 WO의 여러 부품을 분리 확인하고 정비 건수는 유지했다.
- 정지 0은 **7건**. REMARK 빈 문자열 6건과 앞뒤 공백 4건을 그대로 보존했다.
- 최장 대기: **WO-2024-00627 · 12,983분**. 대기시간 전체를 정지 구간으로 해석하지 않는다.
- 최대 정지 길이: **WO-2024-01346 · DOWN_MIN 610분**, 작업시간 1,044분. 두 시간 길이가 다르다.
- 실제 작업 겹침 **2쌍**: WO-2024-01823 / WO-2024-01826 (107분); WO-2024-01859 / WO-2024-01854 (41분).
- 관심 사례 **8개**, 그림 **14개**, 검증 **16개 통과**. 같은 Python 프로세스에서 코드 셀을 순서대로 실제 실행했다.

실행 방식: 로컬 Jupyter 커널은 시작 응답이 없어 자동 실행을 완료하지 못했다. 동일한 프로젝트 Python 환경에서 코드 셀을 순차 실행하여 계산·그림 생성·검증을 확인했다.

### 저장된 결과 그림

- [01_work_table_01.png](../figures/06_maintenance_timeline_jinny/01_work_table_01.png)
- [01_work_table_02.png](../figures/06_maintenance_timeline_jinny/01_work_table_02.png)
- [01_work_table_03.png](../figures/06_maintenance_timeline_jinny/01_work_table_03.png)
- [01_work_table_04.png](../figures/06_maintenance_timeline_jinny/01_work_table_04.png)
- [02_all_maintenance_timeline.png](../figures/06_maintenance_timeline_jinny/02_all_maintenance_timeline.png)
- [03_wait_work_down_comparison.png](../figures/06_maintenance_timeline_jinny/03_wait_work_down_comparison.png)
- [04_case_A.png](../figures/06_maintenance_timeline_jinny/04_case_A.png)
- [04_case_B.png](../figures/06_maintenance_timeline_jinny/04_case_B.png)
- [04_case_C.png](../figures/06_maintenance_timeline_jinny/04_case_C.png)
- [04_case_D.png](../figures/06_maintenance_timeline_jinny/04_case_D.png)
- [04_case_E1.png](../figures/06_maintenance_timeline_jinny/04_case_E1.png)
- [04_case_E2.png](../figures/06_maintenance_timeline_jinny/04_case_E2.png)
- [04_case_F.png](../figures/06_maintenance_timeline_jinny/04_case_F.png)
- [04_case_G.png](../figures/06_maintenance_timeline_jinny/04_case_G.png)